# **1. Instalando pymarc**

In [ ]:
#Librería super importante

!pip install pymarc

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 160.3/160.3 kB 3.8 MB/s eta 0:00:00


# **2. Conectando de google sheet**
### Lectura de las 5 primeras filas

In [ ]:
from pymarc import MARCReader
from google.colab import files
import pandas as pd

# URL del Google Sheet que proporcionaste
google_sheet_url = 'https://docs.google.com/spreadsheets/d/1jjffDCphOIulgG5TYqkT6X81kYS8rMRAoKd-mdgraSY/edit?gid=0#gid=0'

# Convertir la URL para que sea apta para lectura CSV
csv_export_url = google_sheet_url.replace('/edit?gid=', '/export?format=csv&gid=')

try:
    df_sheet = pd.read_csv(csv_export_url)
    print("Content of Google Sheet:")
    display(df_sheet.head())
except Exception as e:
    print(f"Error al leer el Google Sheet: {e}")
    print("Asegúrate de que la hoja de cálculo sea pública o que tu Google Drive esté montado y tengas los permisos adecuados.")


# Este código prepara el archivo MARC para que puedas iterar sobre él y
# acceder a cada uno de los registros bibliográficos que contiene. Para pasar
# de Excel a Marc no es necesario


Content of Google Sheet:


,isbn_020,issn_022,dl_024,fuente_catalogacion_040,idioma_catalogacion_041,cd_dewey_082,autor_persona_100,autor_institucion_110,autor_congreso_111,titulo_245,...,coau_institución_3,koha_tipo_item_942_c,tipo_item_952_y,dewey_952_o,cod_barras_952_p,numero_copia_952_t,numero_copia_952_t.1,numero_copia_952_t.2,952_a,952_b
0,NaN,NaN,NaN,ONPE,NaN,LAM/0001/2026,NaN,Oficina Nacional de Procesos Electorales,NaN,[Láminas informativas 1. Elecciones Generales ...,...,NaN,"trípticos, afiches, flyers","trípticos, afiches, flyers",LAM/0001/2026,NaN,NaN,Ej. 2,Ej. 3,ONPE,ONPE


# **3. Imprimiendo los campos disponibles del primer registro**
### Antes de los nombres de los campos no debe haber espacios de lo contrario no los leerá

In [ ]:
print("Campos del primer registro de Google Sheets:")
# Acceder a la primera fila del DataFrame
first_record = df_sheet.iloc[0]

# Iterar sobre los campos y valores e imprimirlos
for column, value in first_record.items():
    print(f"  {column}: {value}")

Campos del primer registro de Google Sheets:
  isbn_020: nan
  issn_022: nan
  dl_024: nan
  fuente_catalogacion_040: ONPE
  idioma_catalogacion_041: nan
  cd_dewey_082: LAM/0001/2026
  autor_persona_100: nan
  autor_institucion_110: Oficina Nacional de Procesos Electorales
  autor_congreso_111: nan
  titulo_245: [Láminas informativas 1. Elecciones Generales 2026]
  varian_del_título_246: nan
  edicion_250: nan
  lugar_publicacion_260: Lima
  editorial_260: ONPE
  anio_260: 2026
  descripcion_fisica_300: 8 láminas: ilustraciones; 30x42 centímetros
  serie_1_490: nan
  serie_1_490_v: nan
  serie_2_490: nan
  serie_2_490_v: nan
  nota_general_500: nan
  formatted_contents_note_505: Contenido: El domingo 12 de abril de 7.00 a.m. a 5.00 p.m. ¿Cómo elegiremos a nuestras autoridades? ¿Cómo elegirás a los diputados? ¿Cómo elegirás a los senadores? Elige tu local de votación del 23 de noviembre al 14 de diciembre. Si eres miembro de mesa, recuerda que es obligatorio asistir a las 6:30 a.m. Des

# **4. Validando los datos numéricos**
###### ISBN, ISSN DL




In [ ]:
import re
import pandas as pd

def validate_numeric_with_hyphens(value):
    """
    Valida si un valor es numérico, permitiendo guiones.
    Retorna True si es válido, False en caso contrario.
    """
    if pd.isna(value):  # Ignorar valores NaN (celdas vacías)
        return True

    # Convertir a string para manejar diferentes tipos de entrada
    s_value = str(value)

    # Comprobar si el valor resultante contiene solo dígitos y guiones
    # El patrón ^[0-9-]+$ asegura que toda la cadena esté compuesta por dígitos o guiones.
    return bool(re.fullmatch(r'^[0-9-]+$', s_value))


# Columnas a validar
columns_to_validate = ['isbn_020', 'issn_022', 'dl_024']

invalid_entries_found = False

for col_name in columns_to_validate:
    # Verificar si la columna existe en el DataFrame
    if col_name in df_sheet.columns:
        # Aplicar la función de validación a la columna
        # y crear una máscara booleana para los valores inválidos
        invalid_mask = ~df_sheet[col_name].apply(validate_numeric_with_hyphens)

        # Filtrar el DataFrame para mostrar solo las filas con valores inválidos
        invalid_rows = df_sheet[invalid_mask]

        # .empty es un atributo de los DataFrames de Pandas que devuelve True si el DataFrame está vacío
        if not invalid_rows.empty:
            print(f"\nSe encontraron entradas inválidas en la columna '{col_name}':")
            display(invalid_rows[['autor', 'titulo', col_name]]) # Mostrar columnas relevantes para el contexto
            invalid_entries_found = True
    else:
        print(f"Advertencia: La columna '{col_name}' no se encontró en la hoja de cálculo.")

if not invalid_entries_found:
    print("\nTodas las entradas para las columnas 'isbn_020' (ISBN), 'issn_022' (ISSN) y 'dl_024' (DL) son válidas.")
else:
    print("\nPor favor, revisa las entradas inválidas listadas arriba antes de continuar.")


Todas las entradas para las columnas 'isbn_020' (ISBN), 'issn_022' (ISSN) y 'dl_024' (DL) son válidas.


# **5. Pasando csv a mrc**

In [ ]:
import pandas as pd
from pymarc import Record, Field, MARCWriter, Subfield

df = df_sheet # Usamos df_sheet directamente

output_filename = "koha_Libros_from_GoogleSheet.mrc"
writer = MARCWriter(open(output_filename, "wb"))

for i, row in df.iterrows():
    record = Record()

    # 020 ISBN
    if 'isbn_020' in row and pd.notna(row['isbn_020']):
        record.add_field(
            Field(
                tag='020',
                indicators=[' ',' '],
                subfields=[Subfield('a', str(row['isbn_020']))]
            )
        )

    # 022 ISSN
    if 'issn_022' in row and pd.notna(row['issn_022']):
        record.add_field(
            Field(
                tag='022',
                indicators=[' ',' '],
                subfields=[Subfield('a', str(row['issn_022']))]
            )
        )

    # 024 DL
    if 'dl_024' in row and pd.notna(row['dl_024']):
        record.add_field(
            Field(
                tag='024',
                indicators=[' ',' '],
                subfields=[Subfield('a', str(row['dl_024']))]
            )
        )

    # 040 Fuente de Catalogación
    subfields_040 = []
    if 'fuente_catalogacion_040' in row and pd.notna(row['fuente_catalogacion_040']):
        subfields_040.append(Subfield('c', str(row['fuente_catalogacion_040'])))

    if subfields_040:
        record.add_field(
            Field(
                tag='040',
                indicators=[' ',' '],
                subfields=subfields_040
            )
        )

    # 041 Idioma de Catalogación
    if 'idioma_catalogacion_041' in row and pd.notna(row['idioma_catalogacion_041']):
        record.add_field(
            Field(
                tag='041',
                indicators=[' ',' '],
                subfields=[Subfield('a', str(row['idioma_catalogacion_041']))]
            )
        )

    # 082 Clasificación Decimal Dewey
    if 'cd_dewey_082' in row and pd.notna(row['cd_dewey_082']):
        record.add_field(
            Field(
                tag='082',
                indicators=['0','0'],
                subfields=[Subfield('a', str(row['cd_dewey_082']))]
            )
        )

    # 100 Autor Personal
    if 'autor_persona_100' in row and pd.notna(row['autor_persona_100']):
        record.add_field(
            Field(
                tag='100',
                indicators=['1',' '],
                subfields=[Subfield('a', str(row['autor_persona_100']))]
            )
        )

    # 110 Autor Institución
    if 'autor_institucion_110' in row and pd.notna(row['autor_institucion_110']):
        record.add_field(
            Field(
                tag='110',
                indicators=['2',' '],
                subfields=[Subfield('a', str(row['autor_institucion_110']))]
            )
        )

    # 111 Autor Congreso
    if 'autor_congreso_111' in row and pd.notna(row['autor_congreso_111']):
        record.add_field(
            Field(
                tag='111',
                indicators=['2',' '],
                subfields=[Subfield('a', str(row['autor_congreso_111']))]
            )
        )

    # 630 Asiento de Materia - Título Uniforme
    if 'materia_titulo_uniforme_630' in row and pd.notna(row['materia_titulo_uniforme_630']):
        record.add_field(
            Field(
                tag='630',
                indicators=['0','0'],
                subfields=[Subfield('a', str(row['materia_titulo_uniforme_630']))]
            )
        )

    # 245 Título
    if 'titulo_245' in row and pd.notna(row['titulo_245']):
        record.add_field(
            Field(
                tag='245',
                indicators=['1','0'],
                subfields=[Subfield('a', str(row['titulo_245']))]
            )
        )

    # 246 Variantes del Título
    if 'varian_del_título_246' in row and pd.notna(row['varian_del_título_246']):
        record.add_field(
            Field(
                tag='246',
                indicators=['3',' '],
                subfields=[Subfield('a', str(row['varian_del_título_246']))]
            )
        )

    # 250 Mención de Edición
    if 'edicion_250' in row and pd.notna(row['edicion_250']):
        record.add_field(
            Field(
                tag='250',
                indicators=[' ',' '],
                subfields=[Subfield('a', str(row['edicion_250']))]
            )
        )

    # 260 Publicación
    subfields_260 = []
    if 'lugar_publicacion_260' in row and pd.notna(row['lugar_publicacion_260']):
        subfields_260.append(Subfield('a', str(row['lugar_publicacion_260'])))
    if 'editorial_260' in row and pd.notna(row['editorial_260']):
        subfields_260.append(Subfield('b', str(row['editorial_260'])))
    if 'anio_260' in row and pd.notna(row['anio_260']):
        subfields_260.append(Subfield('c', str(int(row['anio_260']))))

    if subfields_260:
        record.add_field(
            Field(
                tag='260',
                indicators=[' ',' '],
                subfields=subfields_260
            )
        )

    # 300 Descripción Física
    if 'descripcion_fisica_300' in row and pd.notna(row['descripcion_fisica_300']):
        record.add_field(
            Field(
                tag='300',
                indicators=[' ',' '],
                subfields=[Subfield('a', str(row['descripcion_fisica_300']))]
            )
        )

    # 490 Mención de Serie 1
    subfields_490_1 = []
    if 'serie_1_490' in row and pd.notna(row['serie_1_490']):
        subfields_490_1.append(Subfield('a', str(row['serie_1_490'])))
    if 'serie_1_490_v' in row and pd.notna(row['serie_1_490_v']):
        subfields_490_1.append(Subfield('v', str(row['serie_1_490_v'])))
    if subfields_490_1:
        record.add_field(
            Field(
                tag='490',
                indicators=['1',' '],
                subfields=subfields_490_1

            )
        )

    # 490 Mención de Serie 2
    subfields_490_2 = []
    if 'serie_2_490' in row and pd.notna(row['serie_2_490']):
        subfields_490_2.append(Subfield('a', str(row['serie_2_490'])))
    if 'serie_2_490_v' in row and pd.notna(row['serie_2_490_v']):
        subfields_490_2.append(Subfield('v', str(row['serie_2_490_v'])))
    if subfields_490_2:
        record.add_field(
                Field(
                    tag='490',
                    indicators=['1',' '],
                    subfields=subfields_490_2
            )
        )


    # 500 Nota General
    if 'nota_general_500' in row and pd.notna(row['nota_general_500']):
        record.add_field(
            Field(
                tag='500',
                indicators=[' ',' '],
                subfields=[Subfield('a', str(row['nota_general_500']))]
            )
        )

    # 505 Formatted Contents Note
    if 'formatted_contents_note_505' in row and pd.notna(row['formatted_contents_note_505']):
        record.add_field(
            Field(
                tag='505',
                indicators=['0',' '],
                subfields=[Subfield('a', str(row['formatted_contents_note_505']))]
            )
        )

    # 520 Resumen
    if 'resumen_520' in row and pd.notna(row['resumen_520']):
        record.add_field(
            Field(
                tag='520',
                indicators=[' ',' '],
                subfields=[Subfield('a', str(row['resumen_520']))]
            )
        )

    # 600 Asiento de Materia - Persona
    for col_suffix in ['1', '2', '3', '4']:
        col_name = f'materia_persona_{col_suffix}_600'
        if col_name in row and pd.notna(row[col_name]):
            record.add_field(
                Field(
                    tag='600',
                    indicators=['1','0'],
                    subfields=[Subfield('a', str(row[col_name]).strip())]
                )
            )

    # 610 Asiento de Materia - Institución
    for col_suffix in ['1', '2', '3', '4']:
        col_name = f'materia_institucion_{col_suffix}_610'
        if col_name in row and pd.notna(row[col_name]):
            record.add_field(
                Field(
                    tag='610',
                    indicators=['2','0'],
                    subfields=[Subfield('a', str(row[col_name]).strip())]
                )
            )

    # 650 Asiento de Materia - Término Temático (Generic)
    for col_suffix in ['1', '2', '3', '4']:
        col_name = f'materia_{col_suffix}'
        if col_name in row and pd.notna(row[col_name]):
            record.add_field(
                Field(
                    tag='650',
                    indicators=[' ','0'],
                    subfields=[Subfield('a', str(row[col_name]).strip())]
                )
            )

    # 651 Asiento de Materia - Término Geográfico
    for col_suffix in ['1', '2']:
        col_name = f'materia_geogra_{col_suffix}'
        if col_name in row and pd.notna(row[col_name]):
            record.add_field(
                Field(
                    tag='651',
                    indicators=[' ','0'],
                    subfields=[Subfield('a', str(row[col_name]).strip())]
                )
            )

    # 655 Término de Indización - Género/Forma
    if 'materia_forma_655' in row and pd.notna(row['materia_forma_655']):
        record.add_field(
            Field(
                tag='655',
                indicators=[' ','7'],
                subfields=[Subfield('a', str(row['materia_forma_655'])), Subfield('2', 'lcgft')]
            )
        )

    # 700 Asiento Secundario - Nombre Personal (coautor_1, coautor_2, coautor_3)
    if 'coautor_1' in row and pd.notna(row['coautor_1']):
        record.add_field(
            Field(
                tag='700',
                indicators=['1',' '],
                subfields=[Subfield('a', str(row['coautor_1'])), Subfield('e', 'coautor')]
            )
        )
    if 'coautor_2' in row and pd.notna(row['coautor_2']):
        record.add_field(
            Field(
                tag='700',
                indicators=['1',' '],
                subfields=[Subfield('a', str(row['coautor_2'])), Subfield('e', 'coautor')]
            )
        )

    if 'coautor_3' in row and pd.notna(row['coautor_3']):
        record.add_field(
            Field(
                tag='700',
                indicators=['1',' '],
                subfields=[Subfield('a', str(row['coautor_3'])), Subfield('e', 'coautor')]
            )
        )

    # 710 Asiento Secundario - Nombre Corporativo (coau_institución_1, etc.)
    for col_suffix in ['1', '2', '3']:
        col_name = f'coau_institución_{col_suffix}'
        if col_name in row and pd.notna(row[col_name]):
            record.add_field(
                Field(
                    tag='710',
                    indicators=['2',' '],
                    subfields=[Subfield('a', str(row[col_name]).strip()), Subfield('e', 'coautor')]
                )
            )

    # 942 Koha Item Type
    if 'koha_tipo_item_942_c' in row and pd.notna(row['koha_tipo_item_942_c']):
        record.add_field(
            Field(
                tag='942',
                indicators=[' ',' '],
                subfields=[Subfield('c', str(row['koha_tipo_item_942_c']))]
            )
        )

    # 952 Ítems (Múltiples para ejemplares)
    # Capturar subcampos comunes para todos los 952
    common_subfields_for_952 = []
    if 'tipo_item_952_y' in row and pd.notna(row['tipo_item_952_y']):
        common_subfields_for_952.append(Subfield('y', str(row['tipo_item_952_y'])))
    if 'dewey_952_o' in row and pd.notna(row['dewey_952_o']):
        common_subfields_for_952.append(Subfield('o', str(row['dewey_952_o'])))
    if 'cod_barras_952_p' in row and pd.notna(row['cod_barras_952_p']):
        # Ensure cod_barras is an integer if present, otherwise append as string
        try:
            common_subfields_for_952.append(Subfield('p', str(int(row['cod_barras_952_p']))))
        except ValueError:
            common_subfields_for_952.append(Subfield('p', str(row['cod_barras_952_p'])))
    if '952_a' in row and pd.notna(row['952_a']):
        common_subfields_for_952.append(Subfield('a', str(row['952_a'])))
    if '952_b' in row and pd.notna(row['952_b']):
        common_subfields_for_952.append(Subfield('b', str(row['952_b'])))

    # Columnas para los números de copia. No se define default_t_value aquí,
    # el comportamiento de 't' vacío se gestiona a continuación.
    copy_columns_to_process = ['numero_copia_952_t', 'numero_copia_952_t.1', 'numero_copia_952_t.2']

    at_least_one_952_generated = False

    for idx, col_name in enumerate(copy_columns_to_process):
        t_subfield_value = None

        if col_name in row and pd.notna(row[col_name]):
            t_subfield_value = str(row[col_name])

        if t_subfield_value is not None:
            # Si hay un valor explícito para 't', se añade un 952 con los subcampos comunes y el 't'
            current_copy_subfields_952 = list(common_subfields_for_952)
            current_copy_subfields_952.append(Subfield('t', t_subfield_value))
            record.add_field(
                Field(
                    tag='952',
                    indicators=[' ',' '],
                    subfields=current_copy_subfields_952
                )
            )
            at_least_one_952_generated = True
        elif idx == 0 and not at_least_one_952_generated and common_subfields_for_952:
            # Este es el caso del primer ejemplar (col_name 'numero_copia_952_t') cuando está vacío,
            # pero existen otros subcampos comunes (y, o, p). Se añade un 952 sin $t.
            record.add_field(
                Field(
                    tag='952',
                    indicators=[' ',' '],
                    subfields=list(common_subfields_for_952)
                )
            )
            at_least_one_952_generated = True

    # Si no se generó ningún 952 y no hay ningún subcampo común, entonces no se añade nada.
    # Si se generó al menos uno, o si el idx==0 y hay common_subfields_for_952, ya está cubierto.


    writer.write(record)

writer.close()

print(f"Archivo MARC '{output_filename}' generado correctamente desde Google Sheet.")

Archivo MARC 'koha_Libros_from_GoogleSheet.mrc' generado correctamente desde Google Sheet.


# **6. Previsualizando archivo mrc generado**

In [ ]:
from pymarc import MARCReader

# Previsualizar los primeros 5 registros MARC generados
print(f"\nPrevisualización de los primeros 5 registros MARC de '{output_filename}':")

records_to_preview = 5
count = 0
with open(output_filename, 'rb') as fh:
    reader = MARCReader(fh, to_unicode=True, force_utf8=True)
    for record in reader:
        if count < records_to_preview:
            print(f"\n--- Registro {count+1} ---")
            print(str(record))
            count += 1
        else:
            break

if count == 0:
    print("No se generaron registros MARC para previsualizar.")


Previsualización de los primeros 5 registros MARC de 'koha_Libros_from_GoogleSheet.mrc':

--- Registro 1 ---
=LDR  01109    a2200217   4500
=040  \\$cONPE
=082  00$aLAM/0001/2026
=110  2\$aOficina Nacional de Procesos Electorales
=245  10$a[Láminas informativas 1. Elecciones Generales 2026]
=260  \\$aLima$bONPE$c2026
=300  \\$a8 láminas: ilustraciones; 30x42 centímetros
=505  0\$aContenido: El domingo 12 de abril de 7.00 a.m. a 5.00 p.m. ¿Cómo elegiremos a nuestras autoridades? ¿Cómo elegirás a los diputados? ¿Cómo elegirás a los senadores? Elige tu local de votación del 23 de noviembre al 14 de diciembre. Si eres miembro de mesa, recuerda que es obligatorio asistir a las 6:30 a.m. Descubre las novedades para las elecciones generales 2026. Conoce las funciones de los diputados.
=600  10$aAutoridades políticas
=650  \0$aElectores
=650  \0$aVoto
=650  \0$aElecciones generales
=650  \0$aPerú
=942  \\$ctrípticos, afiches, flyers
=952  \\$ytrípticos, afiches, flyers$oLAM/0001/2026
=952  \\

# **7. Descargando el archivo mrc**

In [ ]:
from google.colab import files

files.download('koha_Libros_from_GoogleSheet.mrc')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>